# Interstate Border Settlement Analysis in Nigeria
## Seven-Chapter Google Colab Notebook

This notebook identifies settlements located within **5 km on both sides of a shared State/FCT boundary** in Nigeria.

The user selects two States/FCTs. The notebook verifies that they share a border, constructs a 5 km corridor inside each State/FCT, extracts GRID3 settlement extents and names, adds roads, bridges, drainage and health facilities, and produces professional border-focused maps and CSV tables.

### Main outputs
- `interstate_border_settlements_5km.csv`
- `interstate_border_summary.csv`
- `01_interstate_border_multipanel_map.png`
- `02_interstate_border_settlements_map.png`
- ZIP containing the CSV and PNG outputs

### Important interpretation
The 5 km distance is measured in a projected coordinate system from the shared administrative border. It is a geographic screening distance, not travel time. Settlement extents are GRID3 mapped extents and are not official community administrative boundaries.


# Chapter 1 — Prepare the interstate-border analysis environment
Use the same validated administrative, GRID3, OpenStreetMap and health-facility sources already registered in the course GitHub repository.


In [ ]:
# Cell 1 — Install tools, mount Drive, clone the course repository, and load State/FCT lookup
!pip -q install geopandas pyogrio shapely pyproj requests ipywidgets matplotlib mapclassify adjustText osm2geojson

from pathlib import Path
import json, re, subprocess, sys, requests, hashlib, zipfile, shutil
import pandas as pd
import geopandas as gpd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from matplotlib.lines import Line2D
from matplotlib.ticker import FuncFormatter
import ipywidgets as widgets
from IPython.display import display
from google.colab import drive

drive.mount("/content/drive")

REPO = "zubairgis/nigeria-population-demography-course"
RAW = f"https://raw.githubusercontent.com/{REPO}/main"

TEMP_ROOT = Path("/content/interstate_border_course")
TEMP_ROOT.mkdir(parents=True, exist_ok=True)

DRIVE_ROOT = Path("/content/drive/MyDrive/Nigeria_Interstate_Border_Settlements")
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)

REPO_DIR = TEMP_ROOT / "repo"
if REPO_DIR.exists():
    subprocess.run(["git","-C",str(REPO_DIR),"pull","-q"], check=False)
else:
    subprocess.run(["git","clone","-q",f"https://github.com/{REPO}.git",str(REPO_DIR)], check=True)

states = pd.read_csv(
    REPO_DIR/"data/lookups/admin_states_source.csv",
    dtype=str
)

assert len(states) == 37

safe = lambda s: re.sub(r"[^A-Za-z0-9]+","_",str(s)).strip("_")

print("Repository ready:", REPO_DIR)
print("State/FCT choices:", len(states))
print("Persistent output root:", DRIVE_ROOT)


# Chapter 2 — Select two States/FCTs
Choose the two administrative units whose **shared interstate boundary** you want to study. The notebook will stop if the selected pair does not share a border.


In [ ]:
# Cell 2 — Select two States/FCTs
# Rerunning this cell preserves valid selections when possible.

state_options = sorted(
    states["adm1nm"].astype(str).unique()
)

# Recover previous selections from existing widgets or saved variables.
_previous_state1 = (
    state1_dropdown.value
    if "state1_dropdown" in globals()
    and getattr(state1_dropdown, "value", None) in state_options
    else globals().get("STATE1_SELECTION")
)

_previous_state2 = (
    state2_dropdown.value
    if "state2_dropdown" in globals()
    and getattr(state2_dropdown, "value", None) in state_options
    else globals().get("STATE2_SELECTION")
)

_default_1 = (
    _previous_state1
    if _previous_state1 in state_options
    else ("Bayelsa" if "Bayelsa" in state_options else state_options[0])
)

_default_2 = (
    _previous_state2
    if _previous_state2 in state_options
    else ("Delta" if "Delta" in state_options else state_options[1])
)

# Do not allow identical defaults.
if _default_2 == _default_1:
    _default_2 = next(
        x for x in state_options
        if x != _default_1
    )

state1_dropdown = widgets.Dropdown(
    options=state_options,
    value=_default_1,
    description="State 1:"
)

state2_dropdown = widgets.Dropdown(
    options=state_options,
    value=_default_2,
    description="State 2:"
)

def _remember_state_pair(change=None):
    global STATE1_SELECTION, STATE2_SELECTION
    STATE1_SELECTION = state1_dropdown.value
    STATE2_SELECTION = state2_dropdown.value

state1_dropdown.observe(
    _remember_state_pair,
    names="value"
)

state2_dropdown.observe(
    _remember_state_pair,
    names="value"
)

_remember_state_pair()

display(
    widgets.VBox([
        state1_dropdown,
        state2_dropdown
    ])
)

print(
    "Choose two neighbouring States/FCTs above, "
    "then run Cell 3."
)



# Chapter 3 — Define the shared border and the 5 km corridor
The shared border is derived from the SALB administrative geometry already used by the main course notebook.

A **5 km inward corridor** is created inside each selected State/FCT. Together these two zones represent settlements located within 5 km on either side of the interstate boundary.


In [ ]:
# Cell 3 — Load the two State/FCT boundaries, verify adjacency, and build the 5 km corridor

# ------------------------------------------------------------
# RECOVER STATE SELECTIONS IF CELL 2 WAS SKIPPED OR RUNTIME RESET
# ------------------------------------------------------------
state_options = sorted(
    states["adm1nm"].astype(str).unique()
)

if (
    "state1_dropdown" not in globals()
    or "state2_dropdown" not in globals()
):
    print(
        "State-selection widgets were missing. "
        "Recreating them automatically."
    )

    recovered_state1 = globals().get(
        "STATE1_SELECTION",
        "Bayelsa"
    )

    recovered_state2 = globals().get(
        "STATE2_SELECTION",
        "Delta"
    )

    if recovered_state1 not in state_options:
        recovered_state1 = (
            "Bayelsa"
            if "Bayelsa" in state_options
            else state_options[0]
        )

    if (
        recovered_state2 not in state_options
        or recovered_state2 == recovered_state1
    ):
        recovered_state2 = (
            "Delta"
            if "Delta" in state_options
            and "Delta" != recovered_state1
            else next(
                x for x in state_options
                if x != recovered_state1
            )
        )

    state1_dropdown = widgets.Dropdown(
        options=state_options,
        value=recovered_state1,
        description="State 1:"
    )

    state2_dropdown = widgets.Dropdown(
        options=state_options,
        value=recovered_state2,
        description="State 2:"
    )

    display(
        widgets.VBox([
            state1_dropdown,
            state2_dropdown
        ])
    )

STATE1 = state1_dropdown.value
STATE2 = state2_dropdown.value

# Save the active selections for later recovery.
STATE1_SELECTION = STATE1
STATE2_SELECTION = STATE2

print(
    "Using State pair:",
    STATE1,
    "and",
    STATE2
)

if STATE1 == STATE2:
    raise ValueError(
        "Choose two different States/FCTs."
    )

STATE1_ID = states.loc[states.adm1nm.eq(STATE1),"adm1cd"].iloc[0]
STATE2_ID = states.loc[states.adm1nm.eq(STATE2),"adm1cd"].iloc[0]

admin_root = (
    REPO_DIR /
    "data/boundaries/source/BNDA_NGA_2000-01-01_lastupdate/by_state"
)

state1_lgas = gpd.read_file(
    admin_root/f"{STATE1_ID}_{safe(STATE1)}.geojson"
).to_crs(4326)

state2_lgas = gpd.read_file(
    admin_root/f"{STATE2_ID}_{safe(STATE2)}.geojson"
).to_crs(4326)

state1 = state1_lgas.dissolve(by="adm1cd", as_index=False)
state2 = state2_lgas.dissolve(by="adm1cd", as_index=False)

pair = gpd.GeoDataFrame(
    pd.concat([
        state1.assign(state_name=STATE1, state_code=STATE1_ID),
        state2.assign(state_name=STATE2, state_code=STATE2_ID)
    ], ignore_index=True),
    geometry="geometry",
    crs=4326
)

PAIR_CRS = pair.estimate_utm_crs()
if PAIR_CRS is None:
    raise ValueError("Could not determine a projected CRS for the selected State pair.")

s1p = state1.to_crs(PAIR_CRS).geometry.iloc[0]
s2p = state2.to_crs(PAIR_CRS).geometry.iloc[0]

state_distance_m = float(s1p.distance(s2p))

# Exact common boundary from the same SALB source.
shared_border = s1p.boundary.intersection(s2p.boundary)

def _line_only(geom):
    if geom.is_empty:
        return geom
    if geom.geom_type in ["LineString","MultiLineString"]:
        return geom
    if geom.geom_type == "GeometryCollection":
        lines = [
            g for g in geom.geoms
            if g.geom_type in ["LineString","MultiLineString"]
        ]
        if not lines:
            return geom
        return gpd.GeoSeries(lines, crs=PAIR_CRS).union_all()
    return geom

shared_border = _line_only(shared_border)

# Small tolerance fallback for tiny coordinate discrepancies.
if shared_border.is_empty or getattr(shared_border, "length", 0) < 100:
    if state_distance_m > 150:
        raise ValueError(
            f"{STATE1} and {STATE2} do not appear to share a border "
            f"(minimum separation about {state_distance_m/1000:.2f} km)."
        )
    shared_border = _line_only(
        s1p.boundary.intersection(s2p.buffer(75))
    )

if shared_border.is_empty or getattr(shared_border, "length", 0) < 100:
    raise ValueError(
        "A usable shared border line could not be derived. "
        "Choose two States/FCTs that share a land boundary."
    )

BUFFER_M = 5000.0

corridor1_geom = shared_border.buffer(BUFFER_M).intersection(s1p)
corridor2_geom = shared_border.buffer(BUFFER_M).intersection(s2p)
corridor_union_geom = corridor1_geom.union(corridor2_geom)

border_gdf_p = gpd.GeoDataFrame(
    {"border":[f"{STATE1}–{STATE2}"]},
    geometry=[shared_border],
    crs=PAIR_CRS
)

corridors_p = gpd.GeoDataFrame(
    {
        "state":[STATE1,STATE2],
        "state_code":[STATE1_ID,STATE2_ID],
        "side":[1,2]
    },
    geometry=[corridor1_geom,corridor2_geom],
    crs=PAIR_CRS
)

corridors = corridors_p.to_crs(4326)
border_gdf = border_gdf_p.to_crs(4326)

BBOX = tuple(float(x) for x in corridors.total_bounds)

STUDENT_WORK_DIR = (
    DRIVE_ROOT /
    f"{safe(STATE1)}__{safe(STATE2)}"
)
STUDENT_WORK_DIR.mkdir(parents=True, exist_ok=True)

WORK = (
    TEMP_ROOT /
    f"{safe(STATE1)}__{safe(STATE2)}"
)
WORK.mkdir(parents=True, exist_ok=True)

print("State pair:", STATE1, "and", STATE2)
print("Shared border length:", f"{shared_border.length/1000:,.2f} km")
print("Corridor rule: 5 km inside each State/FCT")
print("State 1 corridor area:", f"{corridor1_geom.area/1e6:,.2f} km²")
print("State 2 corridor area:", f"{corridor2_geom.area/1e6:,.2f} km²")
print("Output folder:", STUDENT_WORK_DIR)



# Chapter 4 — Identify settlements within 5 km of the interstate border
GRID3 Nigeria Settlement Extents v3.1 are read from the same GitHub release already registered in the repository. GRID3 settlement-name points are queried using the repository's validated source script.

Settlements are clipped to the appropriate side of the interstate corridor and assigned to the State/FCT and LGA in which the component lies.


In [ ]:
# Cell 4 — Load GRID3 settlement extents and names, then build border-corridor settlement components
CACHE = DRIVE_ROOT/"source_cache"
CACHE.mkdir(parents=True, exist_ok=True)

zip_path = CACHE/"GRID3_NGA_settlement_extents_v3_1_gpkg.zip"
gpkg_path = CACHE/"GRID3_NGA_settlement_extents_v3_1.gpkg"

if not gpkg_path.exists():
    if not zip_path.exists():
        source_cfg = json.loads(
            (REPO_DIR/"config/source_endpoints.json").read_text()
        )
        url = source_cfg["grid3_settlement_extents_v3_1"]["download_url"]
        print("Downloading GRID3 settlement extents once to Drive cache...")
        with requests.get(url, stream=True, timeout=180) as r:
            r.raise_for_status()
            with open(zip_path,"wb") as out:
                for chunk in r.iter_content(chunk_size=1024*1024):
                    if chunk:
                        out.write(chunk)

    with zipfile.ZipFile(zip_path) as z:
        member = [
            n for n in z.namelist()
            if Path(n).name == "GRID3_NGA_settlement_extents_v3_1.gpkg"
        ][0]
        with z.open(member) as src, open(gpkg_path,"wb") as dst:
            shutil.copyfileobj(src,dst)

print("Settlement source:", gpkg_path)

ext = gpd.read_file(
    gpkg_path,
    bbox=BBOX
).to_crs(4326)

required = {"building_count","building_area","type","geometry"}
missing = required - set(ext.columns)
if missing:
    raise ValueError(f"Unexpected GRID3 v3.1 schema; missing: {sorted(missing)}")

# Stable source ID from geometry.
def stable_geom_id(g):
    return "v31_" + hashlib.sha1(g.wkb).hexdigest()[:20]

ext = ext.copy()
ext["source_settlement_id"] = ext.geometry.map(stable_geom_id)
ext["source_building_count"] = pd.to_numeric(
    ext["building_count"], errors="coerce"
)

# Clip the same source settlement layer separately to each side.
components = []
for side_row in corridors.itertuples():
    side_geom = gpd.GeoDataFrame(
        {"state":[side_row.state],"state_code":[side_row.state_code]},
        geometry=[side_row.geometry],
        crs=4326
    )

    comp = gpd.overlay(
        ext,
        side_geom,
        how="intersection",
        keep_geom_type=True
    )

    if len(comp):
        comp["corridor_state"] = side_row.state
        comp["corridor_state_code"] = side_row.state_code
        components.append(comp)

if not components:
    raise ValueError("No GRID3 settlement extents were found in the 5 km border corridor.")

settlements = gpd.GeoDataFrame(
    pd.concat(components, ignore_index=True),
    geometry="geometry",
    crs=4326
)

# Projected geometry metrics.
sp = settlements.to_crs(PAIR_CRS)
source_area_lookup = (
    ext.to_crs(PAIR_CRS)
    .assign(_source_area=lambda d:d.geometry.area)
    .set_index("source_settlement_id")["_source_area"]
    .to_dict()
)

settlements["settlement_area_km2"] = sp.geometry.area.to_numpy()/1e6
settlements["component_fraction_of_source"] = [
    min(1.0, max(0.0, a/source_area_lookup.get(sid,a)))
    for a,sid in zip(sp.geometry.area,settlements["source_settlement_id"])
]
settlements["estimated_component_buildings"] = (
    settlements["source_building_count"] *
    settlements["component_fraction_of_source"]
)

# Distance from each settlement representative point to the shared border.
rep_pts_p = sp.geometry.representative_point()
settlements["distance_to_shared_border_km"] = (
    rep_pts_p.distance(shared_border).to_numpy()/1000.0
)

# Assign LGA by representative point.
all_pair_lgas = gpd.GeoDataFrame(
    pd.concat([state1_lgas,state2_lgas],ignore_index=True),
    geometry="geometry",
    crs=4326
)

rep_wgs = gpd.GeoDataFrame(
    settlements[["source_settlement_id","corridor_state"]].copy(),
    geometry=gpd.GeoSeries(rep_pts_p,crs=PAIR_CRS).to_crs(4326),
    crs=4326
)

lga_join = gpd.sjoin(
    rep_wgs,
    all_pair_lgas[["adm1cd","adm1nm","adm2cd","adm2nm","geometry"]],
    how="left",
    predicate="within"
).drop(columns=["index_right"],errors="ignore")

settlements["lga_code"] = lga_join["adm2cd"].to_numpy()
settlements["lga_name"] = lga_join["adm2nm"].to_numpy()

# Download GRID3 settlement-name points only for the corridor bbox using repository script.
DL = WORK/"downloads"
DL.mkdir(exist_ok=True)

subprocess.run([
    sys.executable,
    str(REPO_DIR/"scripts/01_download_sources.py"),
    "--dataset","settlement_names",
    "--output",str(DL),
    "--bbox",*[str(x) for x in BBOX]
], check=True)

names = gpd.read_file(
    DL/"settlement_names.geojson"
).to_crs(4326)

names = gpd.clip(
    names,
    gpd.GeoDataFrame(geometry=[corridors.geometry.union_all()],crs=4326)
)

# Match names to settlement components.
if len(names):
    nj = gpd.sjoin(
        names,
        settlements[["geometry"]],
        how="inner",
        predicate="within"
    )

    name_field = "set_name" if "set_name" in nj.columns else (
        "name" if "name" in nj.columns else None
    )

    if name_field:
        name_summary = (
            nj.groupby("index_right")[name_field]
            .apply(lambda x:" | ".join(sorted({
                str(v).strip() for v in x.dropna()
                if str(v).strip()
            })))
        )
        settlements["settlement_name"] = settlements.index.map(name_summary)
    else:
        settlements["settlement_name"] = None
else:
    settlements["settlement_name"] = None

# Stable border-analysis serials.
sort_tmp = settlements.to_crs(PAIR_CRS).copy()
rp = sort_tmp.geometry.representative_point()
order = pd.DataFrame({
    "idx":settlements.index,
    "north":rp.y,
    "east":rp.x
}).sort_values(["north","east"],ascending=[False,True])

serial_map = {
    idx:n for n,idx in enumerate(order["idx"],start=1)
}
settlements["settlement_serial"] = [
    serial_map[i] for i in settlements.index
]

print("Settlement components in 5 km + 5 km corridor:",len(settlements))
display(
    settlements[[
        "settlement_serial","corridor_state","lga_name",
        "settlement_name","type","source_building_count",
        "distance_to_shared_border_km"
    ]].head(30)
)


# Chapter 5 — Add roads, bridges, drainage and health facilities
The analysis uses OpenStreetMap roads/bridges, the repository's cached drainage layers, and the same GRID3/NHFR health-facility source used in the main course notebook.


In [ ]:
# Cell 5 — Load roads, bridges, drainage, and health facilities for the 10 km-wide interstate corridor
import osm2geojson

corridor_union_wgs = gpd.GeoDataFrame(
    geometry=[corridors.geometry.union_all()],
    crs=4326
)

minx,miny,maxx,maxy = BBOX

OVERPASS_ENDPOINTS = [
    "https://overpass-api.de/api/interpreter",
    "https://overpass.kumi.systems/api/interpreter",
    "https://overpass.nchc.org.tw/api/interpreter"
]

overpass_query = f"""
[out:json][timeout:180];
(
  way["highway"]({miny},{minx},{maxy},{maxx});
);
out geom;
"""

osm_json = None
for endpoint in OVERPASS_ENDPOINTS:
    try:
        rr = requests.post(
            endpoint,
            data={"data":overpass_query},
            timeout=210,
            headers={"User-Agent":"Hensard-University-Interstate-Border-Course/1.0"}
        )
        rr.raise_for_status()
        osm_json = rr.json()
        print("OSM roads source:",endpoint)
        break
    except Exception as e:
        print("Overpass unavailable:",endpoint,type(e).__name__)

if osm_json is None:
    OSM_ROADS = gpd.GeoDataFrame(
        columns=["highway","name","bridge","geometry"],
        geometry="geometry",crs=4326
    )
else:
    gj = osm2geojson.json2geojson(osm_json)
    OSM_ROADS = gpd.GeoDataFrame.from_features(
        gj["features"],crs=4326
    )

    if len(OSM_ROADS):
        OSM_ROADS = OSM_ROADS[
            OSM_ROADS.geometry.geom_type.isin(["LineString","MultiLineString"])
        ].copy()

        if "tags" in OSM_ROADS.columns:
            def _tag(v,key):
                return v.get(key) if isinstance(v,dict) else None
            for key in ["highway","name","bridge","layer"]:
                if key not in OSM_ROADS.columns:
                    OSM_ROADS[key] = OSM_ROADS["tags"].apply(
                        lambda v,k=key:_tag(v,k)
                    )

        for key in ["highway","name","bridge","layer"]:
            if key not in OSM_ROADS.columns:
                OSM_ROADS[key] = None

        OSM_ROADS = OSM_ROADS[
            OSM_ROADS["highway"].notna()
        ][["highway","name","bridge","layer","geometry"]].copy()

        OSM_ROADS = gpd.clip(
            OSM_ROADS,
            corridor_union_wgs
        )

if len(OSM_ROADS) and "bridge" in OSM_ROADS.columns:
    bv = OSM_ROADS["bridge"].astype(str).str.lower()
    OSM_BRIDGES = OSM_ROADS[
        OSM_ROADS["bridge"].notna() &
        ~bv.isin(["","no","false","0","nan","none"])
    ].copy()
else:
    OSM_BRIDGES = gpd.GeoDataFrame(
        columns=["highway","name","bridge","geometry"],
        geometry="geometry",crs=4326
    )

# Drainage: select repository files from manifest by State code.
drain_manifest = json.loads(
    (REPO_DIR/"data/drainage/manifest.json").read_text()
)

drain_parts = []
for state_code in [STATE1_ID,STATE2_ID]:
    rec = next(
        x for x in drain_manifest["states"]
        if x["adm1cd"] == state_code
    )
    d = gpd.read_file(
        REPO_DIR/"data/drainage"/rec["file"]
    ).to_crs(4326)
    drain_parts.append(d)

DRAINAGE = gpd.GeoDataFrame(
    pd.concat(drain_parts,ignore_index=True),
    geometry="geometry",
    crs=4326
)
DRAINAGE = gpd.clip(DRAINAGE,corridor_union_wgs)

# Health facilities from the same registered source workflow.
subprocess.run([
    sys.executable,
    str(REPO_DIR/"scripts/01_download_sources.py"),
    "--dataset","health_v2",
    "--output",str(DL),
    "--bbox",*[str(x) for x in BBOX]
], check=True)

health = gpd.read_file(
    DL/"health_v2.geojson"
).to_crs(4326)

health = gpd.clip(
    health,
    corridor_union_wgs
)

# Normalize a usable health facility name field.
health_name_candidates = [
    "facility_name","name","facility_n","facilityna","health_fac"
]
health_name_field = next(
    (c for c in health_name_candidates if c in health.columns),
    None
)

if health_name_field and health_name_field != "facility_name":
    health["facility_name"] = health[health_name_field].astype(str)
elif "facility_name" not in health.columns:
    health["facility_name"] = "Health facility"

print("Road features:",len(OSM_ROADS))
print("Bridge features:",len(OSM_BRIDGES))
print("Drainage features:",len(DRAINAGE))
print("Health facilities:",len(health))


# Chapter 6 — Identify border-sharing LGAs and synchronize settlement numbers with Excel
Only LGAs that actually touch the shared State/FCT border are retained. Settlement numbering restarts within each border LGA and the same number is written to the CSV so each map symbol can be traced directly in Excel.

Settlement names remain in the CSV for reference, but the map uses **settlement numbers only**. Place-name labels are handled separately for the LGA panels.


In [ ]:
# Cell 6 — Build border-LGA analysis tables synchronized with map settlement numbers
import numpy as np

# ------------------------------------------------------------
# 1. IDENTIFY ONLY LGAs THAT ACTUALLY SHARE THE STATE BORDER
# ------------------------------------------------------------
all_pair_lgas = gpd.GeoDataFrame(
    pd.concat(
        [state1_lgas, state2_lgas],
        ignore_index=True
    ),
    geometry="geometry",
    crs=4326
)

all_pair_lgas_p = all_pair_lgas.to_crs(
    PAIR_CRS
).copy()

# An LGA is treated as border-sharing when its boundary intersects
# the common State/FCT border (with a small numerical tolerance).
border_lgas_p = all_pair_lgas_p[
    all_pair_lgas_p.geometry.boundary
    .buffer(40)
    .intersects(shared_border)
].copy()

if len(border_lgas_p) == 0:
    raise ValueError(
        "No border-sharing LGAs were detected for the selected State pair."
    )

# Order border LGAs along the shared border so panel order is stable.
border_lgas_p["_touch_geom"] = (
    border_lgas_p.geometry
    .intersection(
        shared_border.buffer(200)
    )
)

border_lgas_p["_touch_pt"] = (
    border_lgas_p["_touch_geom"]
    .representative_point()
)

border_lgas_p["_sort_x"] = (
    border_lgas_p["_touch_pt"].x
)

border_lgas_p["_sort_y"] = (
    border_lgas_p["_touch_pt"].y
)

border_lgas_p = (
    border_lgas_p
    .sort_values(
        ["_sort_x", "_sort_y"],
        ascending=[True, True]
    )
    .reset_index(drop=True)
)

border_lgas_p["panel_order"] = (
    np.arange(
        1,
        len(border_lgas_p) + 1
    )
)

# A is reserved for the shared-border overview.
border_lgas_p["panel_letter"] = [
    chr(66 + i)
    for i in range(
        len(border_lgas_p)
    )
]

border_lgas = (
    border_lgas_p
    .to_crs(4326)
    .copy()
)

print(
    "Border-sharing LGAs detected:",
    len(border_lgas)
)

display(
    border_lgas[
        [
            "panel_order",
            "panel_letter",
            "adm1nm",
            "adm2nm",
            "adm2cd"
        ]
    ]
)

# ------------------------------------------------------------
# 2. KEEP ONLY SETTLEMENT COMPONENTS IN BORDER-SHARING LGAs
# ------------------------------------------------------------
settlements = settlements[
    settlements[
        "lga_code"
    ].astype(str).isin(
        border_lgas[
            "adm2cd"
        ].astype(str)
    )
].copy()

if len(settlements) == 0:
    raise ValueError(
        "No settlement components remain after filtering to border-sharing LGAs."
    )

# Remove any old panel metadata before a rerun.
settlements = settlements.drop(
    columns=[
        "panel_order",
        "panel_letter",
        "panel_lga_name",
        "panel_state_name"
    ],
    errors="ignore"
)

settlements = settlements.merge(
    border_lgas[
        [
            "adm2cd",
            "adm2nm",
            "adm1nm",
            "panel_order",
            "panel_letter"
        ]
    ].rename(
        columns={
            "adm2cd": "lga_code",
            "adm2nm": "panel_lga_name",
            "adm1nm": "panel_state_name"
        }
    ),
    on="lga_code",
    how="left",
    validate="many_to_one"
)

# ------------------------------------------------------------
# 3. NEAREST HEALTH FACILITY DISTANCE — CSV VARIABLE ONLY
# ------------------------------------------------------------
sett_p = settlements.to_crs(
    PAIR_CRS
).copy()

sett_p["_rep_point"] = (
    sett_p.geometry
    .representative_point()
)

health_p = (
    health.to_crs(PAIR_CRS).copy()
    if len(health)
    else health.copy()
)

nearest_health_name = []
nearest_health_km = []

for p in sett_p["_rep_point"]:
    if len(health_p):
        dist = health_p.geometry.distance(p)

        idx = dist.idxmin()

        nearest_health_km.append(
            float(
                dist.loc[idx]
            ) / 1000.0
        )

        nearest_health_name.append(
            str(
                health_p.loc[
                    idx
                ].get(
                    "facility_name",
                    "Health facility"
                )
            )
        )

    else:
        nearest_health_km.append(
            np.nan
        )
        nearest_health_name.append(
            np.nan
        )

sett_p[
    "nearest_health_facility"
] = nearest_health_name

sett_p[
    "nearest_health_distance_km"
] = nearest_health_km

# ------------------------------------------------------------
# 4. CREATE SETTLEMENT NUMBERS SYNCHRONIZED WITH EACH LGA PANEL
# ------------------------------------------------------------
sett_p["_north"] = (
    sett_p["_rep_point"].y
)

sett_p["_east"] = (
    sett_p["_rep_point"].x
)

sett_p = (
    sett_p
    .sort_values(
        [
            "panel_order",
            "_north",
            "_east"
        ],
        ascending=[
            True,
            False,
            True
        ]
    )
    .copy()
)

# Numbering restarts at 1 within each border LGA.
sett_p["lga_settlement_no"] = (
    sett_p
    .groupby("lga_code")
    .cumcount()
    + 1
)

# Also retain a unique overall sequence for data management.
sett_p["global_settlement_no"] = (
    np.arange(
        1,
        len(sett_p) + 1
    )
)

# Representative-point coordinates for Excel.
rep_wgs = gpd.GeoSeries(
    sett_p["_rep_point"],
    crs=PAIR_CRS
).to_crs(
    4326
)

sett_p["longitude"] = (
    rep_wgs.x.to_numpy()
)

sett_p["latitude"] = (
    rep_wgs.y.to_numpy()
)

settlements = (
    sett_p
    .to_crs(4326)
    .drop(
        columns=[
            "_rep_point",
            "_north",
            "_east"
        ],
        errors="ignore"
    )
)

# ------------------------------------------------------------
# 5. PREPARE PLACE LABELS FOR THE LGA PANELS
# ------------------------------------------------------------
# Place names come from the settlement-name point layer loaded in Cell 4.
# These labels are separate from settlement polygon names.
if (
    "names" in globals()
    and len(names)
):
    place_labels = (
        names.to_crs(4326)
        .copy()
    )

    if "set_name" in place_labels.columns:
        place_name_field = "set_name"

    elif "name" in place_labels.columns:
        place_name_field = "name"

    else:
        place_name_field = None

    if place_name_field is not None:
        place_labels[
            "place_label"
        ] = (
            place_labels[
                place_name_field
            ]
            .astype(str)
            .str.strip()
        )

        place_labels = place_labels[
            place_labels[
                "place_label"
            ].notna()
            &
            (
                place_labels[
                    "place_label"
                ] != ""
            )
            &
            (
                place_labels[
                    "place_label"
                ].str.lower()
                != "nan"
            )
        ].copy()

        place_labels = (
            gpd.sjoin(
                place_labels,
                border_lgas[
                    [
                        "adm2cd",
                        "adm2nm",
                        "adm1nm",
                        "geometry"
                    ]
                ],
                how="inner",
                predicate="within"
            )
            .drop(
                columns=[
                    "index_right"
                ],
                errors="ignore"
            )
        )

        # Avoid repeated labels for the same named place within an LGA.
        place_labels = (
            place_labels
            .sort_values(
                "place_label"
            )
            .drop_duplicates(
                subset=[
                    "adm2cd",
                    "place_label"
                ]
            )
            .copy()
        )

    else:
        place_labels = gpd.GeoDataFrame(
            columns=[
                "place_label",
                "adm2cd",
                "adm2nm",
                "adm1nm",
                "geometry"
            ],
            geometry="geometry",
            crs=4326
        )

else:
    place_labels = gpd.GeoDataFrame(
        columns=[
            "place_label",
            "adm2cd",
            "adm2nm",
            "adm1nm",
            "geometry"
        ],
        geometry="geometry",
        crs=4326
    )

# ------------------------------------------------------------
# 6. FINAL CSV TABLES
# ------------------------------------------------------------
settlement_csv = settlements[
    [
        "panel_order",
        "panel_letter",
        "panel_state_name",
        "panel_lga_name",
        "lga_code",
        "lga_settlement_no",
        "global_settlement_no",
        "corridor_state",
        "corridor_state_code",
        "settlement_name",
        "type",
        "source_settlement_id",
        "source_building_count",
        "estimated_component_buildings",
        "component_fraction_of_source",
        "settlement_area_km2",
        "distance_to_shared_border_km",
        "nearest_health_facility",
        "nearest_health_distance_km",
        "longitude",
        "latitude"
    ]
].copy()

settlement_csv = (
    settlement_csv
    .sort_values(
        [
            "panel_order",
            "lga_settlement_no"
        ]
    )
    .reset_index(drop=True)
)

lga_summary_csv = (
    settlement_csv
    .groupby(
        [
            "panel_order",
            "panel_letter",
            "panel_state_name",
            "panel_lga_name",
            "lga_code"
        ],
        dropna=False
    )
    .agg(
        settlement_count=(
            "lga_settlement_no",
            "count"
        ),
        mean_border_distance_km=(
            "distance_to_shared_border_km",
            "mean"
        ),
        mean_health_distance_km=(
            "nearest_health_distance_km",
            "mean"
        ),
        building_count_sum=(
            "source_building_count",
            "sum"
        ),
        estimated_component_buildings=(
            "estimated_component_buildings",
            "sum"
        )
    )
    .reset_index()
)

state_summary_csv = (
    settlement_csv
    .groupby(
        [
            "panel_state_name"
        ],
        dropna=False
    )
    .agg(
        settlement_count=(
            "lga_settlement_no",
            "count"
        ),
        mean_border_distance_km=(
            "distance_to_shared_border_km",
            "mean"
        ),
        mean_health_distance_km=(
            "nearest_health_distance_km",
            "mean"
        ),
        building_count_sum=(
            "source_building_count",
            "sum"
        )
    )
    .reset_index()
)

print(
    "Settlement components in final CSV:",
    len(settlement_csv)
)

print(
    "Border-sharing LGAs:",
    len(lga_summary_csv)
)

display(
    settlement_csv.head(30)
)

display(
    lga_summary_csv.round(2)
)

display(
    state_summary_csv.round(2)
)



# Chapter 7 — Produce dynamic shared-border and LGA-level planning maps
The figure now adapts automatically to the number of LGAs that share the selected interstate border.

- **Panel A:** shared State/FCT border, clipped border-sharing LGAs, LGA labels, roads, and rivers/streams only.
- **Panels B onward:** one panel for each border-sharing LGA, showing settlement polygons, synchronized settlement numbers, roads, drainage, bridges, and place-name labels.
- Settlement-name labels are not drawn on the maps.
- Separate LGA CSV files use the same settlement numbers shown on their corresponding map panel.


In [ ]:
# Cell 7 — Export dynamic border-LGA maps and synchronized CSV files
import math
from adjustText import adjust_text

EXPORT = (
    STUDENT_WORK_DIR /
    "Final_Results"
)

EXPORT.mkdir(
    parents=True,
    exist_ok=True
)

for p in EXPORT.iterdir():
    if p.is_file():
        p.unlink()

# ------------------------------------------------------------
# 1. EXPORT CSV FILES
# ------------------------------------------------------------
settlement_csv.to_csv(
    EXPORT /
    "interstate_border_settlements_5km.csv",
    index=False
)

lga_summary_csv.to_csv(
    EXPORT /
    "interstate_border_lga_summary.csv",
    index=False
)

state_summary_csv.to_csv(
    EXPORT /
    "interstate_border_state_summary.csv",
    index=False
)

# One synchronized CSV for each LGA panel.
for row in lga_summary_csv.itertuples():

    sub = settlement_csv[
        settlement_csv[
            "lga_code"
        ].astype(str).eq(
            str(
                row.lga_code
            )
        )
    ].copy()

    sub.to_csv(
        EXPORT /
        (
            f"{int(row.panel_order):02d}_"
            f"{safe(row.panel_lga_name)}_settlements.csv"
        ),
        index=False
    )

# ------------------------------------------------------------
# 2. PROJECT ALL MAP LAYERS
# ------------------------------------------------------------
border_lgas_map_p = (
    border_lgas
    .to_crs(PAIR_CRS)
    .copy()
)

sett_map_p = (
    settlements
    .to_crs(PAIR_CRS)
    .copy()
)

roads_p = (
    OSM_ROADS
    .to_crs(PAIR_CRS)
    .copy()
    if len(OSM_ROADS)
    else OSM_ROADS.copy()
)

bridges_p = (
    OSM_BRIDGES
    .to_crs(PAIR_CRS)
    .copy()
    if len(OSM_BRIDGES)
    else OSM_BRIDGES.copy()
)

drain_p = (
    DRAINAGE
    .to_crs(PAIR_CRS)
    .copy()
    if len(DRAINAGE)
    else DRAINAGE.copy()
)

place_labels_p = (
    place_labels
    .to_crs(PAIR_CRS)
    .copy()
    if len(place_labels)
    else place_labels.copy()
)

# ------------------------------------------------------------
# 3. OVERVIEW LAYERS
# ------------------------------------------------------------
# The overview shows only clipped border-sharing LGAs,
# shared border, roads and rivers/streams.
overview_clip = (
    shared_border
    .buffer(
        BUFFER_M * 2.0
    )
)

overview_clip_gdf = gpd.GeoDataFrame(
    geometry=[
        overview_clip
    ],
    crs=PAIR_CRS
)

overview_lgas_p = (
    border_lgas_map_p
    .copy()
)

overview_lgas_p[
    "geometry"
] = (
    overview_lgas_p.geometry
    .intersection(
        overview_clip
    )
)

overview_lgas_p = (
    overview_lgas_p[
        ~overview_lgas_p.geometry.is_empty
    ]
    .copy()
)

overview_roads_p = (
    gpd.clip(
        roads_p,
        overview_clip_gdf
    )
    if len(roads_p)
    else roads_p.copy()
)

overview_drain_p = (
    gpd.clip(
        drain_p,
        overview_clip_gdf
    )
    if len(drain_p)
    else drain_p.copy()
)

if (
    len(overview_drain_p)
    and "waterway"
    in overview_drain_p.columns
):
    overview_drain_p = (
        overview_drain_p[
            overview_drain_p[
                "waterway"
            ]
            .fillna("")
            .astype(str)
            .isin(
                [
                    "river",
                    "stream"
                ]
            )
        ]
        .copy()
    )

# ------------------------------------------------------------
# 4. CARTOGRAPHIC STYLES
# ------------------------------------------------------------
STATE_COLORS = {
    STATE1: "#dbe9f6",
    STATE2: "#f7d9d9"
}

SETT_COLORS = {
    "built-up": "#d73027",
    "built_up": "#d73027",
    "urban": "#d73027",
    "city": "#d73027",
    "town": "#fc8d59",
    "hamlet": "#fdae61",
    "village": "#fee08b",
    "small": "#c99700",
    "medium": "#fdd49e",
    "large": "#ef6548",
    "other": "#bdbdbd"
}

ROAD_STYLE = {
    "motorway": (
        "#d73027",
        1.00
    ),
    "trunk": (
        "#f46d43",
        0.90
    ),
    "primary": (
        "#fdae61",
        0.80
    ),
    "secondary": (
        "#fee08b",
        0.65
    ),
    "tertiary": (
        "#8c8c8c",
        0.50
    ),
    "residential": (
        "#bdbdbd",
        0.35
    ),
    "unclassified": (
        "#cccccc",
        0.32
    ),
    "service": (
        "#dddddd",
        0.26
    )
}

DRAIN_STYLE = {
    "river": (
        "#08519c",
        0.85,
        "-"
    ),
    "stream": (
        "#4292c6",
        0.45,
        "-"
    ),
    "canal": (
        "#6baed6",
        0.42,
        "--"
    ),
    "drain": (
        "#9ecae1",
        0.32,
        ":"
    )
}

def classify_type(v):
    s = str(v).lower()

    for k in [
        "built-up",
        "built_up",
        "urban",
        "city",
        "town",
        "hamlet",
        "village",
        "small",
        "medium",
        "large"
    ]:
        if k in s:
            return k

    return "other"


def add_grid(ax):
    ax.grid(
        True,
        linestyle="--",
        linewidth=0.35,
        color="#b0b0b0",
        alpha=0.6
    )

    ax.tick_params(
        labelsize=6
    )

    ax.xaxis.set_major_formatter(
        FuncFormatter(
            lambda x, pos:
                f"{x/1000:.0f}"
        )
    )

    ax.yaxis.set_major_formatter(
        FuncFormatter(
            lambda y, pos:
                f"{y/1000:.0f}"
        )
    )

    ax.set_xlabel(
        "Easting (km)",
        fontsize=7
    )

    ax.set_ylabel(
        "Northing (km)",
        fontsize=7
    )


def add_north(ax):
    ax.annotate(
        "N",
        xy=(
            0.96,
            0.88
        ),
        xytext=(
            0.96,
            0.79
        ),
        xycoords="axes fraction",
        arrowprops=dict(
            facecolor="black",
            width=1.6,
            headwidth=6
        ),
        ha="center",
        va="center",
        fontsize=8,
        fontweight="bold"
    )


def add_scale(ax):
    x0, x1 = ax.get_xlim()
    y0, y1 = ax.get_ylim()

    width = x1 - x0
    height = y1 - y0

    choices = np.array(
        [
            1000,
            2000,
            5000,
            10000,
            20000,
            50000
        ],
        dtype=float
    )

    length = float(
        choices[
            np.argmin(
                np.abs(
                    choices -
                    width * 0.14
                )
            )
        ]
    )

    sx = (
        x0 +
        width * 0.045
    )

    sy = (
        y0 +
        height * 0.018
    )

    ax.plot(
        [
            sx,
            sx + length
        ],
        [
            sy,
            sy
        ],
        color="black",
        linewidth=2.0,
        zorder=50
    )

    ax.plot(
        [
            sx,
            sx
        ],
        [
            sy - height * 0.005,
            sy + height * 0.005
        ],
        color="black",
        linewidth=1.0,
        zorder=50
    )

    ax.plot(
        [
            sx + length,
            sx + length
        ],
        [
            sy - height * 0.005,
            sy + height * 0.005
        ],
        color="black",
        linewidth=1.0,
        zorder=50
    )

    ax.text(
        sx + length / 2,
        sy + height * 0.010,
        f"{length/1000:g} km",
        ha="center",
        va="bottom",
        fontsize=6
    )


def draw_roads(
    ax,
    roads_layer,
    thin=False
):
    if not len(
        roads_layer
    ):
        return

    factor = (
        0.70
        if thin
        else 1.00
    )

    if (
        "highway"
        not in roads_layer.columns
    ):
        roads_layer.plot(
            ax=ax,
            color="#bdbdbd",
            linewidth=0.35,
            zorder=5
        )
        return

    for hw, grp in roads_layer.groupby(
        roads_layer[
            "highway"
        ]
        .fillna(
            "unclassified"
        )
        .astype(str)
    ):
        color, lw = ROAD_STYLE.get(
            hw,
            (
                "#cccccc",
                0.32
            )
        )

        grp.plot(
            ax=ax,
            color=color,
            linewidth=max(
                lw * factor,
                0.18
            ),
            zorder=5
        )


def draw_drainage(
    ax,
    drain_layer
):
    if not len(
        drain_layer
    ):
        return

    if (
        "waterway"
        not in drain_layer.columns
    ):
        drain_layer.plot(
            ax=ax,
            color="#4292c6",
            linewidth=0.40,
            zorder=4
        )
        return

    for w, grp in drain_layer.groupby(
        drain_layer[
            "waterway"
        ]
        .fillna(
            "stream"
        )
        .astype(str)
    ):
        color, lw, ls = (
            DRAIN_STYLE.get(
                w,
                (
                    "#4292c6",
                    0.40,
                    "-"
                )
            )
        )

        grp.plot(
            ax=ax,
            color=color,
            linewidth=lw,
            linestyle=ls,
            zorder=4
        )

# ------------------------------------------------------------
# 5. DYNAMIC MULTI-PANEL FIGURE
# ------------------------------------------------------------
n_lga_panels = (
    len(
        border_lgas_map_p
    )
)

n_panels_total = (
    1 +
    n_lga_panels
)

ncols = 2

nrows = math.ceil(
    n_panels_total /
    ncols
)

fig, axes = plt.subplots(
    nrows,
    ncols,
    figsize=(
        18,
        6.5 * nrows
    )
)

axes = (
    np.array(
        axes
    )
    .reshape(-1)
)

# ------------------------------------------------------------
# PANEL A — SHARED BORDER + BORDER-SHARING LGAs
# ------------------------------------------------------------
ax = axes[0]

for state_name, grp in overview_lgas_p.groupby(
    "adm1nm"
):
    grp.plot(
        ax=ax,
        facecolor=STATE_COLORS.get(
            state_name,
            "#efefef"
        ),
        edgecolor="#666666",
        linewidth=0.8,
        alpha=0.85,
        zorder=1
    )

gpd.GeoSeries(
    [
        shared_border
    ],
    crs=PAIR_CRS
).plot(
    ax=ax,
    color="black",
    linewidth=2.0,
    zorder=10
)

# Overview: roads + rivers/streams only.
draw_roads(
    ax,
    overview_roads_p,
    thin=False
)

draw_drainage(
    ax,
    overview_drain_p
)

# LGA labels only.
for _, r in overview_lgas_p.iterrows():

    p = (
        r.geometry
        .representative_point()
    )

    ax.text(
        p.x,
        p.y,
        str(
            r[
                "adm2nm"
            ]
        ),
        fontsize=7,
        fontweight="bold",
        ha="center",
        va="center",
        color="black"
    )

xmin_a, ymin_a, xmax_a, ymax_a = (
    overview_lgas_p.total_bounds
)

padx_a = max(
    (
        xmax_a -
        xmin_a
    ) * 0.04,
    1000
)

pady_a = max(
    (
        ymax_a -
        ymin_a
    ) * 0.07,
    1000
)

ax.set_xlim(
    xmin_a - padx_a,
    xmax_a + padx_a
)

ax.set_ylim(
    ymin_a - pady_a,
    ymax_a + pady_a
)

add_grid(ax)
add_north(ax)
add_scale(ax)

ax.set_title(
    "A. Shared Border and Border-Sharing LGAs",
    fontsize=12,
    fontweight="bold"
)

handles_a = [
    Patch(
        facecolor=STATE_COLORS.get(
            STATE1,
            "#dbe9f6"
        ),
        edgecolor="#666666",
        label=STATE1
    ),
    Patch(
        facecolor=STATE_COLORS.get(
            STATE2,
            "#f7d9d9"
        ),
        edgecolor="#666666",
        label=STATE2
    ),
    Line2D(
        [0],
        [0],
        color="black",
        lw=2.0,
        label="Shared State border"
    ),
    Line2D(
        [0],
        [0],
        color="#fdae61",
        lw=1.0,
        label="Road"
    ),
    Line2D(
        [0],
        [0],
        color="#08519c",
        lw=1.0,
        label="River / stream"
    )
]

ax.legend(
    handles=handles_a,
    loc="lower right",
    fontsize=6,
    title="Legend",
    title_fontsize=7,
    ncol=2
)

# ------------------------------------------------------------
# PANELS B... — ONE PANEL PER BORDER-SHARING LGA
# ------------------------------------------------------------
for i, lga_row in enumerate(
    border_lgas_map_p.itertuples(),
    start=1
):
    ax = axes[i]

    lga_geom = (
        lga_row.geometry
    )

    lga_code = str(
        lga_row.adm2cd
    )

    lga_name = str(
        lga_row.adm2nm
    )

    state_name = str(
        lga_row.adm1nm
    )

    panel_letter = str(
        lga_row.panel_letter
    )

    lga_gdf = gpd.GeoDataFrame(
        {
            "name": [
                lga_name
            ]
        },
        geometry=[
            lga_geom
        ],
        crs=PAIR_CRS
    )

    sett_lga = (
        sett_map_p[
            sett_map_p[
                "lga_code"
            ].astype(str).eq(
                lga_code
            )
        ]
        .copy()
    )

    roads_lga = (
        gpd.clip(
            roads_p,
            lga_gdf
        )
        if len(roads_p)
        else roads_p.copy()
    )

    bridges_lga = (
        gpd.clip(
            bridges_p,
            lga_gdf
        )
        if len(bridges_p)
        else bridges_p.copy()
    )

    drain_lga = (
        gpd.clip(
            drain_p,
            lga_gdf
        )
        if len(drain_p)
        else drain_p.copy()
    )

    places_lga = (
        gpd.clip(
            place_labels_p,
            lga_gdf
        )
        if len(place_labels_p)
        else place_labels_p.copy()
    )

    # Base LGA polygon.
    lga_gdf.plot(
        ax=ax,
        facecolor=STATE_COLORS.get(
            state_name,
            "#f2f2f2"
        ),
        edgecolor="#444444",
        linewidth=1.0,
        alpha=0.35,
        zorder=1
    )

    # Shared-border segment in this LGA.
    border_seg = (
        shared_border
        .intersection(
            lga_geom.buffer(
                50
            )
        )
    )

    if not border_seg.is_empty:
        gpd.GeoSeries(
            [
                border_seg
            ],
            crs=PAIR_CRS
        ).plot(
            ax=ax,
            color="black",
            linewidth=1.8,
            zorder=9
        )

    # Roads, drainage and bridges.
    draw_drainage(
        ax,
        drain_lga
    )

    draw_roads(
        ax,
        roads_lga,
        thin=True
    )

    if len(
        bridges_lga
    ):
        bridges_lga.plot(
            ax=ax,
            color="#111111",
            linewidth=0.60,
            zorder=8
        )

    # Settlement polygons + synchronized settlement numbers only.
    if len(
        sett_lga
    ):
        sett_lga[
            "_type_std"
        ] = (
            sett_lga[
                "type"
            ]
            .apply(
                classify_type
            )
        )

        for typ, grp in sett_lga.groupby(
            "_type_std"
        ):
            grp.plot(
                ax=ax,
                facecolor=SETT_COLORS.get(
                    typ,
                    "#bdbdbd"
                ),
                edgecolor="none",
                alpha=0.90,
                zorder=3
            )

        rep_pts = (
            sett_lga.geometry
            .representative_point()
        )

        texts = []

        for j, (_, row) in enumerate(
            sett_lga.iterrows()
        ):
            p = rep_pts.iloc[j]

            texts.append(
                ax.text(
                    p.x,
                    p.y,
                    str(
                        int(
                            row[
                                "lga_settlement_no"
                            ]
                        )
                    ),
                    fontsize=4.2,
                    ha="center",
                    va="center",
                    color="black",
                    zorder=12
                )
            )

        if len(
            texts
        ) <= 250:
            adjust_text(
                texts,
                ax=ax,
                expand=(
                    1.04,
                    1.10
                ),
                force_text=(
                    0.22,
                    0.32
                ),
                ensure_inside_axes=True,
                max_move=(
                    10,
                    10
                )
            )

    # PLACE labels only.
    # Settlement polygon names are deliberately NOT labelled.
    place_texts = []

    if len(
        places_lga
    ):
        for _, row in places_lga.iterrows():

            nm = str(
                row.get(
                    "place_label",
                    ""
                )
            ).strip()

            if (
                nm
                and
                nm.lower()
                not in [
                    "nan",
                    "none"
                ]
            ):
                p = (
                    row.geometry
                )

                place_texts.append(
                    ax.text(
                        p.x,
                        p.y,
                        nm,
                        fontsize=4.2,
                        ha="left",
                        va="bottom",
                        color="black",
                        zorder=15
                    )
                )

        if len(
            place_texts
        ) <= 120:
            adjust_text(
                place_texts,
                ax=ax,
                expand=(
                    1.02,
                    1.10
                ),
                force_text=(
                    0.18,
                    0.28
                ),
                ensure_inside_axes=True,
                max_move=(
                    10,
                    10
                )
            )

    xmin, ymin, xmax, ymax = (
        lga_geom.bounds
    )

    padx = max(
        (
            xmax -
            xmin
        ) * 0.05,
        600
    )

    pady = max(
        (
            ymax -
            ymin
        ) * 0.05,
        600
    )

    ax.set_xlim(
        xmin - padx,
        xmax + padx
    )

    ax.set_ylim(
        ymin - pady,
        ymax + pady
    )

    add_grid(ax)
    add_north(ax)
    add_scale(ax)

    ax.set_title(
        (
            f"{panel_letter}. "
            f"{lga_name} "
            f"({state_name})"
        ),
        fontsize=11,
        fontweight="bold"
    )

# Hide unused axes.
for k in range(
    n_panels_total,
    len(axes)
):
    axes[k].axis(
        "off"
    )

# Common legend for LGA panels.
legend_handles = [
    Patch(
        facecolor=SETT_COLORS[
            "built-up"
        ],
        label="Built-up"
    ),
    Patch(
        facecolor=SETT_COLORS[
            "hamlet"
        ],
        label="Hamlet"
    ),
    Patch(
        facecolor=SETT_COLORS[
            "small"
        ],
        label="Small / village"
    ),
    Line2D(
        [0],
        [0],
        color="#fdae61",
        lw=0.9,
        label="Road"
    ),
    Line2D(
        [0],
        [0],
        color="#111111",
        lw=0.8,
        label="Bridge"
    ),
    Line2D(
        [0],
        [0],
        color="#08519c",
        lw=0.9,
        label="River"
    ),
    Line2D(
        [0],
        [0],
        color="#4292c6",
        lw=0.8,
        label="Stream"
    ),
    Line2D(
        [0],
        [0],
        color="black",
        lw=1.8,
        label="Shared State border"
    )
]

fig.legend(
    handles=legend_handles,
    loc="lower center",
    ncol=4,
    fontsize=8,
    frameon=True,
    bbox_to_anchor=(
        0.5,
        0.01
    )
)

fig.suptitle(
    (
        "Settlements within 5 km of the Shared "
        f"{STATE1}–{STATE2} Border"
    ),
    fontsize=17,
    fontweight="bold",
    y=0.98
)

fig.tight_layout(
    rect=[
        0.02,
        0.05,
        0.98,
        0.95
    ]
)

multipanel_png = (
    EXPORT /
    "01_interstate_border_dynamic_lga_panels.png"
)

fig.savefig(
    multipanel_png,
    dpi=300,
    bbox_inches="tight",
    facecolor="white"
)

plt.close(
    fig
)

# ------------------------------------------------------------
# 6. CLEAN SHARED-BORDER OVERVIEW MAP
# ------------------------------------------------------------
fig, ax = plt.subplots(
    figsize=(
        15,
        7
    )
)

for state_name, grp in overview_lgas_p.groupby(
    "adm1nm"
):
    grp.plot(
        ax=ax,
        facecolor=STATE_COLORS.get(
            state_name,
            "#efefef"
        ),
        edgecolor="#555555",
        linewidth=0.9,
        alpha=0.85,
        zorder=1
    )

gpd.GeoSeries(
    [
        shared_border
    ],
    crs=PAIR_CRS
).plot(
    ax=ax,
    color="black",
    linewidth=2.0,
    zorder=10
)

draw_roads(
    ax,
    overview_roads_p,
    thin=False
)

draw_drainage(
    ax,
    overview_drain_p
)

for _, r in overview_lgas_p.iterrows():

    p = (
        r.geometry
        .representative_point()
    )

    ax.text(
        p.x,
        p.y,
        str(
            r[
                "adm2nm"
            ]
        ),
        fontsize=8,
        fontweight="bold",
        ha="center",
        va="center",
        color="black"
    )

ax.set_xlim(
    xmin_a - padx_a,
    xmax_a + padx_a
)

ax.set_ylim(
    ymin_a - pady_a,
    ymax_a + pady_a
)

add_grid(ax)
add_north(ax)
add_scale(ax)

ax.set_title(
    (
        "Border-Sharing LGAs along the "
        f"{STATE1}–{STATE2} Shared Border"
    ),
    fontsize=14,
    fontweight="bold"
)

overview_png = (
    EXPORT /
    "02_interstate_border_overview_lgas.png"
)

fig.savefig(
    overview_png,
    dpi=300,
    bbox_inches="tight",
    facecolor="white"
)

plt.close(
    fig
)

# ------------------------------------------------------------
# 7. ZIP OUTPUTS
# ------------------------------------------------------------
zip_path = (
    STUDENT_WORK_DIR /
    (
        f"{safe(STATE1)}_"
        f"{safe(STATE2)}_"
        "interstate_border_lga_outputs.zip"
    )
)

if zip_path.exists():
    zip_path.unlink()

files_to_zip = [
    p.name
    for p in EXPORT.iterdir()
    if p.suffix.lower()
    in {
        ".csv",
        ".png"
    }
]

subprocess.run(
    [
        "zip",
        "-q",
        str(
            zip_path
        ),
        *files_to_zip
    ],
    cwd=EXPORT,
    check=True
)

print(
    "Outputs saved in:",
    EXPORT
)

for p in sorted(
    EXPORT.iterdir()
):
    print(
        " -",
        p.name
    )

print(
    "ZIP:",
    zip_path
)

from google.colab import files
files.download(
    str(
        zip_path
    )
)


